In [1]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import XSD, OWL, RDFS, SKOS, DC
import pandas as pd

In [2]:
# create graph
g = Graph()
# g.parse("") # load semantic model and/or ontologies

# header
onto = URIRef("https://placeholder.pt/test.owl")
g.add((onto, RDF.type, OWL.Ontology))
g.add((onto, DC.title, Literal("Test Knowledge Graph")))
g.add((onto, DC.creator, Literal("Creator 1, Creator 2")))

<Graph identifier=Na5e9f968b7794fe7b9a274996d09b48e (<class 'rdflib.graph.Graph'>)>

In [3]:
# create new namespaces
OBOINOWL = Namespace("http://www.geneontology.org/formats/oboInOwl#")
BASE = Namespace("https://placeholder.pt/")

# bind namespaces for better readability
g.bind("owl", OWL)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("skos", SKOS)
g.bind("dc", DC)
g.bind("oboInOwl", OBOINOWL)

In [4]:
pub_class = URIRef(BASE + "publication")

In [5]:
# converts table file into a dataframe
def open_df(file):
    df = pd.read_csv(file, sep=",", header=0)
    df.dropna(how='all', axis=1, inplace=True) # remove empty columns
    df.dropna(how='all', axis=0, inplace=True) # remove empty rows
    return df

In [6]:
# creates publication instances from PubMed table
def create_publications(file):
    df = open_df(file)
    for idx, row in df.iterrows():
        # iri based on PMID from table
        uri = URIRef(BASE + "PMID_" + str(row["PMID"]))
        # create instance of publication class
        g.add((uri, RDF.type, pub_class))
        # add info from table
        g.add((uri, URIRef(BASE + "has_PMID"), Literal(row["PMID"], datatype=XSD.integer)))
        g.add((uri, URIRef(BASE + "has_authors"), Literal(row["Authors"], datatype=XSD.string)))
        g.add((uri, URIRef(BASE + "published_in_journal"), Literal(row["Journal"], datatype=XSD.string)))
        g.add((uri, URIRef(BASE + "publication_year"), Literal(row["Year"], datatype=XSD.integer)))

        if not pd.isnull(row["PMCID"]):
            g.add((uri, URIRef(BASE + "has_PMCID"), Literal(row["PMCID"], datatype=XSD.string)))
        # if not pd.isnull(row["NIHMS ID"]):
        #     g.add((uri, URIRef(BASE + "has_NIHMS_ID"), Literal(row["NIHMS ID"], datatype=XSD.string)))
        if not pd.isnull(row["DOI"]):
            g.add((uri, URIRef(BASE + "DOI"), Literal(row["DOI"], datatype=XSD.string)))

In [7]:
create_publications("./PubMed_Research/PubMed_Complete_Report.csv")

In [8]:
# save graph in OWL format
g.serialize(destination="./graph_with-data.owl", format="xml")

<Graph identifier=Na5e9f968b7794fe7b9a274996d09b48e (<class 'rdflib.graph.Graph'>)>